# 📊 Notebook 2 — Exploratory Data Analysis

We explore spending patterns, income trends, and category breakdowns visually.

**Questions we answer:**
1. How did income and expenses change month to month?
2. Where does most of the money go?
3. Which categories are eating the most budget?
4. How does salary compare to side income?
5. What is the savings rate each month?

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

tx  = pd.read_csv('../data/cleaned.csv', parse_dates=['date'])
bud = pd.read_csv('../data/budgets.csv')
income   = tx[tx['type']=='Income'].copy()
expenses = tx[tx['type']=='Expense'].copy()
print('Data loaded ✅  —', len(tx), 'transactions')

## 1 — Monthly Income vs Expenses

Side-by-side bar chart showing total income and total expenses per month. The gap between them is net savings.

In [ ]:
monthly_income   = income.groupby(['month','month_name'])['amount'].sum().reset_index().rename(columns={'amount':'total_income'})
monthly_expenses = expenses.groupby(['month','month_name'])['amount'].sum().reset_index().rename(columns={'amount':'total_expenses'})
monthly = monthly_income.merge(monthly_expenses, on=['month','month_name'])
monthly['net_savings']  = monthly['total_income'] - monthly['total_expenses']
monthly['savings_rate'] = (monthly['net_savings'] / monthly['total_income'] * 100).round(1)
monthly = monthly.sort_values('month')

fig = px.bar(monthly, x='month_name', y=['total_income','total_expenses'],
             barmode='group', title='Monthly Income vs Expenses',
             labels={'value':'Amount (R)','variable':'','month_name':'Month'},
             color_discrete_sequence=['#2ecc71','#e74c3c'],
             template='plotly_dark')
fig.show()
monthly[['month_name','total_income','total_expenses','net_savings','savings_rate']]

## 2 — Salary vs Side Income Breakdown

Shows how much of total income came from the salary versus freelance work each month.

In [ ]:
sal  = income[income['category']=='Salary'].groupby('month_name')['amount'].sum().reset_index().rename(columns={'amount':'Salary'})
side = income[income['category']=='Side Income'].groupby('month_name')['amount'].sum().reset_index().rename(columns={'amount':'Side Income'})
inc_breakdown = sal.merge(side, on='month_name', how='left').fillna(0)
month_order = ['January','February','March']
inc_breakdown['month_name'] = pd.Categorical(inc_breakdown['month_name'], categories=month_order, ordered=True)
inc_breakdown = inc_breakdown.sort_values('month_name')

fig = px.bar(inc_breakdown, x='month_name', y=['Salary','Side Income'],
             barmode='stack', title='Income Breakdown — Salary vs Side Income',
             labels={'value':'Amount (R)','variable':'Source','month_name':'Month'},
             color_discrete_sequence=['#3498db','#f39c12'],
             template='plotly_dark')
fig.show()

total_side = income[income['category']=='Side Income']['amount'].sum()
total_income = income['amount'].sum()
print(f'Side income as % of total income: {total_side/total_income*100:.1f}%')

## 3 — Spending by Category (All 3 Months)

Pie chart and bar chart showing where money was spent across the full period.

In [ ]:
cat_totals = expenses.groupby('category')['amount'].sum().reset_index().sort_values('amount', ascending=False)

fig = px.pie(cat_totals, values='amount', names='category',
             title='Total Spending by Category (Jan–Mar 2024)',
             template='plotly_dark',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

fig2 = px.bar(cat_totals, x='category', y='amount',
              title='Total Spending per Category',
              labels={'amount':'Total Spent (R)','category':'Category'},
              color='amount', color_continuous_scale='Oranges',
              template='plotly_dark')
fig2.update_layout(xaxis_tickangle=-30)
fig2.show()

## 4 — Daily Transaction Timeline

Each transaction plotted over time coloured by category — useful for spotting spending spikes.

In [ ]:
fig = px.scatter(expenses, x='date', y='amount',
                 color='category', size='amount',
                 title='Daily Expense Transactions (Jan–Mar 2024)',
                 labels={'amount':'Amount (R)','date':'Date'},
                 template='plotly_dark',
                 size_max=20)
fig.show()

## 5 — Savings Rate Trend

In [ ]:
fig = px.bar(monthly, x='month_name', y='savings_rate',
             title='Monthly Savings Rate (%)',
             labels={'savings_rate':'Savings Rate (%)','month_name':'Month'},
             color='savings_rate', color_continuous_scale='RdYlGn',
             text='savings_rate', template='plotly_dark')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.add_hline(y=20, line_dash='dash', line_color='white',
              annotation_text='Target: 20%')
fig.show()

print('Savings Rate Summary:')
for _, row in monthly.iterrows():
    status = '✅' if row['savings_rate'] >= 20 else '⚠️ '
    print(f'  {status} {row["month_name"]}: {row["savings_rate"]}%  (net R {row["net_savings"]:,.0f})')